# Hosted Tools

Function tools ran as plain Python functions in _your_ process. This notebook
covers the SDK's hosted tools — `WebSearchTool`, `FileSearchTool`,
`CodeInterpreterTool`, and `ImageGenerationTool` — which execute entirely on
OpenAI's own infrastructure as part of a real API call.

Worth saying up front: because hosted tools run server-side, there's no
client-side function for `ScriptedModel` to intercept — there's no offline way to
prove one actually searched the web or ran code. This notebook verifies what
_can_ be verified offline (construction and config), and is explicit about what
needs a live key.


## Construct all four hosted tools

Each just needs its own config — `WebSearchTool` takes none required,
`FileSearchTool` needs `vector_store_ids`, and `CodeInterpreterTool`/
`ImageGenerationTool` need a `tool_config` dict.


In [ ]:
from agents import (
    CodeInterpreterTool,  # lets the model run Python code on OpenAI's servers
    FileSearchTool,  # searches through files you uploaded to a vector store
    ImageGenerationTool,  # lets the model make images on OpenAI's servers
    WebSearchTool,  # lets the model search the web on OpenAI's servers
)

web_search_tool = WebSearchTool(
    # search_context_size: how much of each web page the model gets back
    search_context_size="low",
)

file_search_tool = FileSearchTool(
    # vector_store_ids: which uploaded vector stores to search
    vector_store_ids=["vs_leaf_village_records"],  # placeholder: use a real vector store ID here
    # max_num_results: the most matching pieces that can come back
    max_num_results=3,
)

code_interpreter_tool = CodeInterpreterTool(
    # tool_config: container="auto" lets OpenAI set up the sandbox for you
    tool_config={
        "type": "code_interpreter",
        "container": {"type": "auto"},
    }
)

image_generation_tool = ImageGenerationTool(
    # tool_config: quality and size change the cost and size of the made image
    tool_config={
        "type": "image_generation",
        "quality": "low",
        "size": "1024x1024",
    }
)

# proves all four hosted tool types can be made on your computer with only their own settings.
# No API call happens when they are made
for tool in [
    web_search_tool,
    file_search_tool,
    code_interpreter_tool,
    image_generation_tool,
]:
    print("[constructed]", type(tool).__name__)

## Attach them to one agent

Constructing an `Agent` with hosted tools is entirely local — no API call
happens until you actually `Runner.run` it.


In [ ]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
)

# Initialize the agent
leaf_research_agent = Agent(
    name="Leaf Village Research Assistant",
    instructions=(
        "Use web search for real-world context, file search for Village records, "
        "code interpreter for calculations, and image generation for mission reports."
    ),
    # tools: hosted tools can be mixed with function tools in the same list
    tools=[
        web_search_tool,
        file_search_tool,
        code_interpreter_tool,
        image_generation_tool,
    ],
)
# proves hosted tools go into Agent(tools=...) just like function tools do,
# and that making the Agent still does not call the API
print("[tools]", [type(tool).__name__ for tool in leaf_research_agent.tools])

## Run it live — optional, needs a real key

The only way to see a hosted tool actually execute — uses just `WebSearchTool`,
since the others need a real vector store/container this notebook doesn't
provision.


**Doc-drift note (installed `agents==0.22.3`):** `WebSearchTool` calls never
produce a `tool_call_output_item`. Introspecting
`agents.run_internal.turn_resolution` shows a `ResponseFunctionWebSearch`
output only ever gets wrapped as a `ToolCallItem`
(`items.append(ToolCallItem(raw_item=output, agent=agent))`) — there's no
matching output item appended anywhere in the run loop. `FileSearchTool`
results are handled identically. This makes sense once you look at the raw
item's shape (`id`, `action`, `status` — no result payload): the search runs
entirely server-side and the model consumes the results internally, folding
them straight into its response text, so there's nothing separate to hand
back to the client as an "output." `tool_call_output_item` is reserved for
tool types that return a discrete payload to the SDK — function tools,
computer-use actions, local/hosted shell calls, apply_patch/program output.
The cell below checks `tool_call_item` instead and prints the search
action/status from `raw_item`.


In [ ]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
    WebSearchTool,  # lets the model search the web on OpenAI's servers
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: the only way to see a hosted tool really run, using a real WebSearchTool call with the real API
async def demo_live() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the agent
    web_only_agent = Agent(
        name="Leaf Village Research Assistant",
        instructions="Use web search to answer briefly.",
        # tools: the tools this agent is allowed to call
        tools=[WebSearchTool(search_context_size="low")],
    )

    # Run the Agent
    live_result = await Runner.run(
        web_only_agent,
        "Marvel movie Avengers Dooms Day release date in India?",
        # run_config: settings for just this run
        run_config=RunConfig(workflow_name="e04-live-demo"),
    )

    for item in live_result.new_items:
        if item.type == "tool_call_item":
            print("[tool_call_item]", item.raw_item.action, item.raw_item.status)
    print("[live]", live_result.final_output)


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

## Summary

- Hosted tools execute server-side — there's no local callable for
  `ScriptedModel` to intercept, so proving they _work_ needs a live key.
- What's fully verifiable offline is correct construction, required config
  fields, and that they attach to an `Agent` correctly.
